# Regional Credit Default Risk Mapping

**Question:** Does adding geographic risk information improve loan default prediction, and where are the highest-risk regions?

A null result is a valid outcome here. If borrower features already capture what geography would tell us, the geographic feature will not improve the model — report that honestly rather than tuning until it looks better.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)

RAW = "../data/raw"
OUT = "../outputs"
RANDOM_STATE = 42

## 1. Load data

Download `accepted_2007_to_2018Q4.csv.gz` from [Kaggle](https://www.kaggle.com/datasets/wordsforthewise/lending-club) into `data/raw/`.

The full file is millions of rows. Load only the columns needed, and optionally subset by year to keep this tractable on a laptop.

In [ ]:
COLS = [
    "loan_status",   # target
    "addr_state",    # geography
    "issue_d",       # for time filtering
    "loan_amnt",
    "term",
    "int_rate",
    "installment",
    "grade",
    "sub_grade",
    "emp_length",
    "home_ownership",
    "annual_inc",
    "verification_status",
    "purpose",
    "dti",
    "delinq_2yrs",
    "open_acc",
    "pub_rec",
    "revol_bal",
    "revol_util",
    "total_acc",
]

df = pd.read_csv(f"{RAW}/accepted_2007_to_2018Q4.csv.gz", usecols=COLS, low_memory=False)
print(f"Loaded {len(df):,} rows")
df.head()

## 2. Define the target

Only **resolved** loans have a known outcome. Loans still `Current` have not defaulted *yet* — including them as non-default would label future defaults as good loans and bias the model optimistic. Drop them.

In [ ]:
print(df["loan_status"].value_counts())

GOOD = ["Fully Paid"]
BAD = ["Charged Off", "Default"]

df = df[df["loan_status"].isin(GOOD + BAD)].copy()
df["default"] = df["loan_status"].isin(BAD).astype(int)

print(f"\nResolved loans: {len(df):,}")
print(f"Default rate: {df['default'].mean():.2%}")

In [ ]:
# Optional: restrict to a recent vintage for speed and consistency.
df["issue_d"] = pd.to_datetime(df["issue_d"], format="%b-%Y", errors="coerce")
df = df[df["issue_d"].dt.year.between(2015, 2018)].copy()
print(f"After year filter: {len(df):,} rows, default rate {df['default'].mean():.2%}")

## 3. Clean features

In [ ]:
# term: " 36 months" -> 36
df["term"] = df["term"].str.extract(r"(\d+)").astype(float)

# emp_length: "10+ years" -> 10, "< 1 year" -> 0
df["emp_length"] = (
    df["emp_length"]
    .replace({"10+ years": "10", "< 1 year": "0"})
    .str.extract(r"(\d+)")
    .astype(float)
)

# int_rate / revol_util may carry % signs depending on the export
for col in ["int_rate", "revol_util"]:
    if df[col].dtype == object:
        df[col] = df[col].str.rstrip("%").astype(float)

df = df.dropna(subset=["addr_state", "annual_inc", "dti"])
print(f"After cleaning: {len(df):,} rows")
df.describe()

## 4. Train/test split

Split **before** computing any state-level statistics. State default rates computed on all data would leak test outcomes into the training features.

In [ ]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(
    df, test_size=0.2, stratify=df["default"], random_state=RANDOM_STATE
)
print(f"Train {len(train):,}  |  Test {len(test):,}")
print(f"Default rate — train {train['default'].mean():.2%}, test {test['default'].mean():.2%}")

## 5. Geographic exploration

Default rate by state, computed on the training split only.

In [ ]:
state_stats = (
    train.groupby("addr_state")["default"]
    .agg(default_rate="mean", n_loans="size")
    .sort_values("default_rate", ascending=False)
)

# States with very few loans have noisy rates — flag them rather than trusting the number.
print("Highest default rates (states with >= 500 training loans):")
print(state_stats[state_stats["n_loans"] >= 500].head(10))
print("\nLowest:")
print(state_stats[state_stats["n_loans"] >= 500].tail(10))

In [ ]:
import geopandas as gpd

# US Census cartographic boundary file (public domain)
GEOJSON = "https://raw.githubusercontent.com/PublicaMundi/MappingAPI/master/data/geojson/us-states.json"
states_gdf = gpd.read_file(GEOJSON)

# The GeoJSON uses full state names; LendingClub uses 2-letter codes.
STATE_ABBREV = {
    "Alabama": "AL", "Alaska": "AK", "Arizona": "AZ", "Arkansas": "AR", "California": "CA",
    "Colorado": "CO", "Connecticut": "CT", "Delaware": "DE", "District of Columbia": "DC",
    "Florida": "FL", "Georgia": "GA", "Hawaii": "HI", "Idaho": "ID", "Illinois": "IL",
    "Indiana": "IN", "Iowa": "IA", "Kansas": "KS", "Kentucky": "KY", "Louisiana": "LA",
    "Maine": "ME", "Maryland": "MD", "Massachusetts": "MA", "Michigan": "MI", "Minnesota": "MN",
    "Mississippi": "MS", "Missouri": "MO", "Montana": "MT", "Nebraska": "NE", "Nevada": "NV",
    "New Hampshire": "NH", "New Jersey": "NJ", "New Mexico": "NM", "New York": "NY",
    "North Carolina": "NC", "North Dakota": "ND", "Ohio": "OH", "Oklahoma": "OK", "Oregon": "OR",
    "Pennsylvania": "PA", "Rhode Island": "RI", "South Carolina": "SC", "South Dakota": "SD",
    "Tennessee": "TN", "Texas": "TX", "Utah": "UT", "Vermont": "VT", "Virginia": "VA",
    "Washington": "WA", "West Virginia": "WV", "Wisconsin": "WI", "Wyoming": "WY",
}
states_gdf["addr_state"] = states_gdf["name"].map(STATE_ABBREV)

geo = states_gdf.merge(state_stats.reset_index(), on="addr_state", how="left")
print(f"Matched {geo['default_rate'].notna().sum()} of {len(geo)} state geometries")

In [ ]:
# Continental US view (drop AK/HI so the map isn't dominated by empty ocean)
conus = geo[~geo["addr_state"].isin(["AK", "HI"])]

fig, ax = plt.subplots(figsize=(14, 8))
conus.plot(
    column="default_rate",
    cmap="OrRd",
    linewidth=0.5,
    edgecolor="0.7",
    legend=True,
    legend_kwds={"label": "Default rate", "shrink": 0.6},
    missing_kwds={"color": "lightgrey", "label": "No data"},
    ax=ax,
)
ax.set_title("Loan default rate by state (2015-2018 vintages, training split)", fontsize=14)
ax.axis("off")
plt.tight_layout()
plt.savefig(f"{OUT}/default_rate_by_state.png", dpi=150, bbox_inches="tight")
plt.show()

### Optional: overlay economic context

State unemployment rates from FRED (series `<STATE>UR`, e.g. `CAUR`). Skip if you would rather keep scope tight — the modeling section does not depend on it.

If you do add it: a correlation between state unemployment and state default rate is **descriptive, not causal**. Say it that way in the writeup.

## 6. Baseline model — borrower features only

No geographic information at all. This is the bar the geographic model has to beat.

In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score
from xgboost import XGBClassifier

NUMERIC = [
    "loan_amnt", "term", "int_rate", "installment", "emp_length", "annual_inc",
    "dti", "delinq_2yrs", "open_acc", "pub_rec", "revol_bal", "revol_util", "total_acc",
]
CATEGORICAL = ["grade", "home_ownership", "verification_status", "purpose"]


def build_features(frame, extra_cols=None):
    """One-hot encode categoricals, keep numerics, optionally append extra columns."""
    X = frame[NUMERIC].copy()
    dummies = pd.get_dummies(frame[CATEGORICAL], drop_first=True)
    X = pd.concat([X, dummies], axis=1)
    if extra_cols:
        for col in extra_cols:
            X[col] = frame[col].values
    return X


X_train = build_features(train)
X_test = build_features(test)
# Align columns in case a category appears in only one split
X_train, X_test = X_train.align(X_test, join="left", axis=1, fill_value=0)

y_train, y_test = train["default"], test["default"]

baseline = XGBClassifier(
    n_estimators=400,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
baseline.fit(X_train, y_train)

p_base = baseline.predict_proba(X_test)[:, 1]
auc_base = roc_auc_score(y_test, p_base)
ap_base = average_precision_score(y_test, p_base)
print(f"Baseline  ROC-AUC {auc_base:.4f}  |  PR-AUC {ap_base:.4f}")

## 7. Add the geographic feature

State default rate is computed **on the training split only**, then mapped onto both splits. States that appear in test but not train fall back to the training global mean.

This is the step where leakage would normally creep in — computing the rate on the full dataset would let test-set outcomes influence the feature.

In [ ]:
train_state_rate = train.groupby("addr_state")["default"].mean()
global_mean = train["default"].mean()

train = train.assign(state_risk=train["addr_state"].map(train_state_rate).fillna(global_mean))
test = test.assign(state_risk=test["addr_state"].map(train_state_rate).fillna(global_mean))

X_train_geo = build_features(train, extra_cols=["state_risk"])
X_test_geo = build_features(test, extra_cols=["state_risk"])
X_train_geo, X_test_geo = X_train_geo.align(X_test_geo, join="left", axis=1, fill_value=0)

geo_model = XGBClassifier(
    n_estimators=400,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
geo_model.fit(X_train_geo, y_train)

p_geo = geo_model.predict_proba(X_test_geo)[:, 1]
auc_geo = roc_auc_score(y_test, p_geo)
ap_geo = average_precision_score(y_test, p_geo)

print(f"Baseline     ROC-AUC {auc_base:.4f}  |  PR-AUC {ap_base:.4f}")
print(f"+ geography  ROC-AUC {auc_geo:.4f}  |  PR-AUC {ap_geo:.4f}")
print(f"\nDelta        ROC-AUC {auc_geo - auc_base:+.4f}  |  PR-AUC {ap_geo - ap_base:+.4f}")

### Is the difference real, or noise?

A delta of +0.001 AUC on a single split is not a finding. Re-run both models across several random seeds and compare the spread — if the improvement is smaller than the seed-to-seed variation, geography did not help and the honest conclusion is a null result.

In [ ]:
def run_seed(seed):
    """Full pipeline on one seed; returns (baseline_auc, geo_auc)."""
    tr, te = train_test_split(df, test_size=0.2, stratify=df["default"], random_state=seed)

    rate = tr.groupby("addr_state")["default"].mean()
    gmean = tr["default"].mean()
    tr = tr.assign(state_risk=tr["addr_state"].map(rate).fillna(gmean))
    te = te.assign(state_risk=te["addr_state"].map(rate).fillna(gmean))

    params = dict(
        n_estimators=400, max_depth=5, learning_rate=0.05, subsample=0.8,
        colsample_bytree=0.8, eval_metric="logloss", random_state=seed, n_jobs=-1,
    )

    results = []
    for extra in (None, ["state_risk"]):
        Xtr = build_features(tr, extra_cols=extra)
        Xte = build_features(te, extra_cols=extra)
        Xtr, Xte = Xtr.align(Xte, join="left", axis=1, fill_value=0)
        m = XGBClassifier(**params).fit(Xtr, tr["default"])
        results.append(roc_auc_score(te["default"], m.predict_proba(Xte)[:, 1]))
    return results


seed_results = [run_seed(s) for s in [0, 1, 2, 3, 4]]
arr = np.array(seed_results)

print(f"Baseline    AUC  mean {arr[:, 0].mean():.4f}  std {arr[:, 0].std():.4f}")
print(f"+ geography AUC  mean {arr[:, 1].mean():.4f}  std {arr[:, 1].std():.4f}")
print(f"\nMean improvement: {(arr[:, 1] - arr[:, 0]).mean():+.4f}")
print(f"Seed-to-seed std: {arr[:, 0].std():.4f}")
print("\nIf the improvement is smaller than the seed-to-seed std, treat it as noise.")

## 8. SHAP — did geography actually matter?

Where does `state_risk` rank against borrower features? If it sits near the bottom, that is the explanation for a null result: borrower-level features already encode what geography would have told us.

In [ ]:
import shap

sample = X_test_geo.sample(min(5000, len(X_test_geo)), random_state=RANDOM_STATE)
explainer = shap.TreeExplainer(geo_model)
shap_values = explainer.shap_values(sample)

shap.summary_plot(shap_values, sample, plot_type="bar", max_display=15, show=False)
plt.title("Feature importance (mean |SHAP|)")
plt.tight_layout()
plt.savefig(f"{OUT}/shap_importance.png", dpi=150, bbox_inches="tight")
plt.show()

mean_abs = pd.Series(np.abs(shap_values).mean(axis=0), index=sample.columns).sort_values(ascending=False)
rank = list(mean_abs.index).index("state_risk") + 1
print(f"state_risk ranks {rank} of {len(mean_abs)} features by mean |SHAP|")

In [ ]:
shap.summary_plot(shap_values, sample, max_display=15, show=False)
plt.tight_layout()
plt.savefig(f"{OUT}/shap_beeswarm.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Findings

Fill this in with the actual numbers. Template:

- Default rate varies from **[X]%** to **[Y]%** across states (restricted to states with >= 500 loans)
- Baseline ROC-AUC **[  ]**, with geographic feature **[  ]** — a change of **[  ]** against seed-to-seed variation of **[  ]**
- `state_risk` ranked **[  ]** of **[  ]** features by mean |SHAP|
- **Conclusion:** [did geography add signal, or was it already captured by borrower features?]

### A note on fair lending

Geographic risk features in real credit models carry redlining risk — location correlates with protected characteristics, so a model that prices loans by region can reproduce historical discrimination even when race is never an input. This project is an academic exercise on public data, not a deployable scoring system. Worth being able to say that out loud in an interview: it shows you understand why risk teams have model governance.